In [1]:
import requests
import pandas as pd
from tqdm import tqdm
import sqlite3

In [2]:
url = "https://api.tcgdex.net/v2/en/cards"

response_all_cards = requests.get(url, timeout=15)
response_all_cards.raise_for_status()

response_all_cards_json  = response_all_cards.json()


In [3]:
print("Total number of cards:", len(response_all_cards_json))
print("Sample card data:", response_all_cards_json[3])

Total number of cards: 23160
Sample card data: {'id': 'swsh9-001', 'localId': '001', 'name': 'Exeggcute', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001'}


In [4]:
def get_card_details(card_id):
    url = f"https://api.tcgdex.net/v2/en/cards/{card_id}"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    return response.json()
card_details = get_card_details(response_all_cards_json[3]['id'])
print("Card details:", card_details)

Card details: {'category': 'Pokemon', 'id': 'swsh9-001', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001', 'localId': '001', 'name': 'Exeggcute', 'rarity': 'Common', 'set': {'cardCount': {'official': 172, 'total': 216}, 'id': 'swsh9', 'logo': 'https://assets.tcgdex.net/en/swsh/swsh9/logo', 'name': 'Brilliant Stars', 'symbol': 'https://assets.tcgdex.net/univ/swsh/swsh9/symbol'}, 'variants': {'firstEdition': False, 'holo': False, 'normal': True, 'reverse': True, 'wPromo': False}, 'variants_detailed': [{'type': 'normal', 'size': 'standard', 'variantId': 'generated'}, {'type': 'reverse', 'size': 'standard', 'variantId': 'generated'}], 'dexId': [102], 'hp': 50, 'types': ['Grass'], 'stage': 'Basic', 'attacks': [{'cost': ['Colorless'], 'name': 'Ram', 'damage': 10}, {'cost': ['Grass', 'Colorless'], 'name': 'Seed Bomb', 'damage': 20}], 'retreat': 1, 'regulationMark': 'F', 'legal': {'standard': False, 'expanded': True}, 'updated': '2025-08-16T20:39:55Z', 'pricing': {'cardmarket': {'updated

In [5]:
card_details["set"]

{'cardCount': {'official': 172, 'total': 216},
 'id': 'swsh9',
 'logo': 'https://assets.tcgdex.net/en/swsh/swsh9/logo',
 'name': 'Brilliant Stars',
 'symbol': 'https://assets.tcgdex.net/univ/swsh/swsh9/symbol'}

In [6]:
# JSON (dict) -> DataFrame "piatto" (1 riga)
df_card = pd.json_normalize(card_details, sep="_")
display(df_card)

,category,id,image,localId,name,rarity,variants_detailed,dexId,hp,types,...,pricing_cardmarket_avg1,pricing_cardmarket_avg7,pricing_cardmarket_avg30,pricing_cardmarket_avg-holo,pricing_cardmarket_low-holo,pricing_cardmarket_trend-holo,pricing_cardmarket_avg1-holo,pricing_cardmarket_avg7-holo,pricing_cardmarket_avg30-holo,pricing_tcgplayer
0,Pokemon,swsh9-001,https://assets.tcgdex.net/en/swsh/swsh9/001,001,Exeggcute,Common,"[{'type': 'normal', 'size': 'standard', 'varia...",[102],50,[Grass],...,0.02,0.03,0.03,0.16,0.02,0.09,0.05,0.19,0.16,None


In [26]:
all_df_cards = []
for card in tqdm(response_all_cards_json[:300]):
    try:
        details = get_card_details(card['id'])
        df_card = pd.json_normalize(details, sep="_")
        df_card["espansione_id"] = details["set"]["id"]
        df_card["espansione_nome"] = details["set"]["name"]
        if "image" not in details: 
            df_card["image"] = [None]
        if not details["pricing"]["cardmarket"]:
            df_card["pricing_cardmarket_low"] = [0]
        df_card = df_card[["id", "name","espansione_id", "espansione_nome", "pricing_cardmarket_low", "image"]]
        all_df_cards.append(df_card)
    except Exception as e:
        print(f"Error fetching details for card {card['id']}: {e}")
        #break

  1%|          | 2/300 [00:00<02:22,  2.09it/s]

Error fetching details for card exu-%3F: 404 Client Error: Not Found for url: https://api.tcgdex.net/v2/en/cards/exu-%3F


100%|██████████| 300/300 [01:47<00:00,  2.80it/s]


In [32]:
import random
df_all = pd.concat(all_df_cards, ignore_index=True)
df_test = df_all.sample(15)[["id", "name","espansione_id", "espansione_nome"]].copy()
cards_condizioni = ["Mint", "Near Mint", "Excellent", "Good", "Light Played", "Played", "Poor"]
df_test["condizione"] = [random.choice(cards_condizioni) for _ in range(len(df_test))]
df_test["prezzo"] = [round(random.uniform(1, 100), 2) for _ in range(len(df_test))]
df_test["quantita_stock"] = [random.randint(1, 5) for _ in range(len(df_test))]
df_test["prezzo_acquisto"] = [round(random.uniform(0, 1) * prezzo, 2)  for prezzo in df_test["prezzo"]]
df_test

C:\Users\s.galati\AppData\Local\Temp\ipykernel_31276\2677101736.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_df_cards, ignore_index=True)


,id,name,espansione_id,espansione_nome,condizione,prezzo,quantita_stock,prezzo_acquisto
149,sve-001,Grass Energy,sve,Scarlet & Violet Energy,Near Mint,90.22,3,20.85
190,sm10-1,Pheromosa & Buzzwole GX,sm10,Unbroken Bonds,Played,59.74,5,53.13
74,sm8-1,Tangela,sm8,Lost Thunder,Excellent,22.83,1,17.23
219,xy0-2,Pansage,xy0,Kalos Starter Set,Good,82.40,4,1.29
164,sv10.5w-001,Sewaddle,sv10.5w,White Flare,Excellent,29.99,4,12.82
80,dp5-1,Articuno,dp5,Majestic Dawn,Near Mint,33.04,2,27.71
247,bw1-2,Snivy,bw1,Black & White,Excellent,53.39,5,43.93
205,swsh10-002,Hisuian Voltorb,swsh10,Astral Radiance,Good,47.26,4,0.19
120,pl1-1,Ampharos,pl1,Platinum,Near Mint,26.21,2,5.11
4,swsh11-001,Oddish,swsh11,Lost Origin,Excellent,47.19,4,3.42


In [33]:
conn = sqlite3.connect("pokemon.db")
cursor = conn.cursor()
df_test.to_sql(
    "stock",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()

In [34]:
conn = sqlite3.connect("../card_database.db")
cursor = conn.cursor()

df_all = pd.concat(all_df_cards, ignore_index=True)
df_all.to_sql(
    "DatabaseCards",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)

C:\Users\s.galati\AppData\Local\Temp\ipykernel_31276\3787411377.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_df_cards, ignore_index=True)


299